In [1]:
from read_training_data import get_tension_data, get_raw_data, get_lengths, calculate_res_freq
import pandas as pd
import numpy as np

In [2]:
raw_data_path = "training-data/raw/APA_5947a8b0-5152-11ed-84c6-9170f34dd096b_apa4_training"
processed_data_path = "training-data/processed/5947a8b0-5152-11ed-84c6-9170f34dd096b_apa4_training.json"

In [3]:
proc_scans = get_tension_data(processed_data_path, ["U", "X"])


In [4]:
keys_to_remove = {"scanEndMode", "measuredBy", "stage", "apaUuid", "type"}
raw_scans = get_raw_data(raw_data_path, keys_to_remove)

In [5]:
proc_df = pd.DataFrame(proc_scans)
raw_df = pd.DataFrame(raw_scans)
# raw_df.head()

In [6]:
comb_df = pd.merge(raw_df, proc_df, on="uid", how="outer")
comb_df.head(3)

,uid,segment_id,channel_index,freq,ampl,layer,headboardNumber,side,parent_folder,stage,tension
0,UA1002,1002,0,"[50.0, 50.125, 50.25, 50.375, 50.5, 50.625, 50...","[3870.976801, 3870.854905, 3875.409452, 3883.0...",U,5,A,training-data/raw/APA_5947a8b0-5152-11ed-84c6-...,Winding,6.164400
1,UA1004,1004,1,"[50.0, 50.125, 50.25, 50.375, 50.5, 50.625, 50...","[1328.26138, 1327.855029, 1329.099351, 1332.22...",U,5,A,training-data/raw/APA_5947a8b0-5152-11ed-84c6-...,Winding,6.331951
2,UA1006,1006,2,"[50.0, 50.125, 50.25, 50.375, 50.5, 50.625, 50...","[3779.794773, 3780.85742, 3783.779545, 3791.47...",U,5,A,training-data/raw/APA_5947a8b0-5152-11ed-84c6-...,Winding,6.381051


In [7]:
final_comb_df = comb_df.dropna(subset=['tension'])
numeric_tension = pd.to_numeric(final_comb_df['tension'], errors='coerce') # removes tensions that aren't found
final_comb_df['tension'] = numeric_tension
final_comb_df.shape

(611, 11)

In [8]:
u_side_lengths = get_lengths("training-data/tensions/U_Layer_Tension_Template.xlsx", "U")
x_side_lengths = get_lengths("training-data/tensions/X_Layer_Tension_Template.xlsx", "X")

In [9]:
final_df = pd.merge(final_comb_df, u_side_lengths, on="uid", how="inner") # this only includes U layer wires


In [ ]:
x_wires = final_comb_df[final_comb_df["uid"].str.startswith("X", na=False)]
x_wires["length"] = x_side_lengths
x_wires.head()

In [11]:
final_df = pd.concat([final_df, x_wires], ignore_index=True) # reincludes X layers
final_df.shape

(611, 12)

In [12]:
tension = final_df['tension'].values
lengths = final_df['length'].values / 1000 # in meters

rho = 1.52e-4
 

res_freq = calculate_res_freq(tension, lengths, rho) # in Hz

print(res_freq.shape)
final_df['res_freq'] = res_freq

(611,)


In [13]:
final_df[final_df["uid"].str.startswith("X", na=False)].head(3)

,uid,segment_id,channel_index,freq,ampl,layer,headboardNumber,side,parent_folder,stage,tension,length,res_freq
534,XA18,18,7,"[64.0, 64.125, 64.25, 64.375, 64.5, 64.625, 64...","[1529.835389, 1529.242702, 1531.671658, 1535.8...",X,1,A,training-data/raw/APA_5947a8b0-5152-11ed-84c6-...,Winding,6.004333,1273.0,78.064251
535,XA20,20,6,"[64.0, 64.125, 64.25, 64.375, 64.5, 64.625, 64...","[4554.047346, 4555.735923, 4561.981086, 4571.3...",X,1,A,training-data/raw/APA_5947a8b0-5152-11ed-84c6-...,Winding,6.386445,1273.0,80.509918
536,XA22,22,5,"[64.0, 64.125, 64.25, 64.375, 64.5, 64.625, 64...","[1552.343604, 1553.119515, 1555.380363, 1557.3...",X,1,A,training-data/raw/APA_5947a8b0-5152-11ed-84c6-...,Winding,6.444849,1273.0,80.877216


In [14]:
# validating the resonant frequency calculation
final_df[(final_df['res_freq'] < 50)].sort_values(by='res_freq', ascending=False)[["parent_folder", "uid", "res_freq"]].head(5)

,parent_folder,uid,res_freq
27,training-data/raw/APA_5947a8b0-5152-11ed-84c6-...,UA143,48.837003
168,training-data/raw/APA_5947a8b0-5152-11ed-84c6-...,UA943,46.465810
100,training-data/raw/APA_5947a8b0-5152-11ed-84c6-...,UA543,46.429169


In [15]:
x=final_df[final_df["uid"].str.startswith("X", na=False)].iloc[1]
print(max(x['freq']))

112.00000028


In [1]:
from read_training_data import get_tension_data, get_raw_data, get_lengths, calculate_res_freq
import pandas as pd
import numpy as np

In [2]:
# testing new dataset
new_raw_path = "/vols/dune/amh24/APA-DWA-measurements/4ml/scanData/raw/APA_APA30_PostCold_27_01_26/"
new_proc_path = "/vols/dune/amh24/APA-DWA-measurements/4ml/scanData/processed/APA30_PostCold_27_01_26.json"

In [3]:
proc_scans = get_tension_data(new_proc_path, stage="DWA Development")
keys_to_remove = {"scanEndMode", "measuredBy", "apaUuid", "type"}
raw_scans = get_raw_data(new_raw_path, keys_to_remove)

In [4]:
proc_df = pd.DataFrame(proc_scans)
raw_df = pd.DataFrame(raw_scans)

In [5]:
proc_df.head(3)

,uid,tension
0,GA4,7.12
1,GA6,6.98
2,GA7,6.56


In [13]:
print(proc_df.shape)
print(proc_df["uid"].unique().shape)

(5518, 2)
(5518,)


In [16]:
raw_df.shape

(8000, 10)

In [ ]:
comb_df = pd.merge(raw_df, proc_df, on="uid", how="outer")
final_comb_df = comb_df.dropna(subset=['tension'])
final_comb_df.head(4)

In [15]:
final_comb_df.shape

(6564, 11)

In [23]:
final_comb_df[final_comb_df.duplicated(subset=['uid'], keep=False)]["uid"].value_counts()

uid
GB439    10
GB441    10
GB443    10
GB445    10
GB447    10
         ..
XB439     2
XB441     2
XB443     2
XB445     2
XB447     2
Name: count, Length: 462, dtype: int64

In [24]:
# temp solution in order to test rest of code
unique_df = final_comb_df.drop_duplicates(subset=['uid'], keep='first') 

In [26]:
u_side_lengths = get_lengths("training-data/tensions/U_Layer_Tension_Template.xlsx", "U")
v_side_lengths = get_lengths("training-data/tensions/V_Layer_Tension_Template.xlsx", "V")
x_side_lengths = get_lengths("training-data/tensions/X_Layer_Tension_Template.xlsx", "X")
g_side_lengths = get_lengths("training-data/tensions/G_Layer_Tension_Template.xlsx", "G")

In [35]:
full_u_df = pd.merge(unique_df, u_side_lengths, on="uid", how="inner") # this only includes U layer wires
full_v_df = pd.merge(unique_df, v_side_lengths, on="uid", how="inner")

full_x_df = unique_df[unique_df["uid"].str.startswith("X", na=False)]
full_x_df["length"] = x_side_lengths

full_g_df = unique_df[unique_df["uid"].str.startswith("G", na=False)]
full_g_df["length"] = g_side_lengths

In [36]:
df_list = [full_u_df, full_v_df, full_x_df, full_g_df]
final_df = pd.concat(df_list, ignore_index=True)

In [38]:
print(final_df.shape)
print(unique_df.shape)

(5518, 12)
(5518, 11)


In [40]:
tension = final_df['tension'].values
lengths = final_df['length'].values / 1000 # in meters

rho = 1.52e-4
 

res_freq = calculate_res_freq(tension, lengths, rho) # in Hz

print(res_freq.shape)
final_df['res_freq'] = res_freq

(5518,)


/home/hep/sb3423/msci-project/DWA-ML/initial-data-cleaning/read_training_data.py:206: RuntimeWarning: invalid value encountered in sqrt
  return np.sqrt(tensions / rho) * (1 / (2*lengths))
